# TDWI Lab 3 Part 3: PR Review Automations

In this lesson you will configure a **comments-only Cursor Automation** for PR review, run a **Cloud Agent** to add a Streamlit **Revenue Explorer**, walk through draft → ready → automation comments → optional fix agent on the **same branch**, merge to `main`, then add **`scripts/check.sh`** and the **pre-PR harness** (deterministic checks + sub-agent review).

**Lab design note:** You are learning the **agent-as-reviewer** pattern and how **Automations** wire it on GitHub. Cursor also ships **Bugbot** and **Approval Agents**; other tools offer equivalents (e.g. `/babysit` for PR follow-up). In production, explore those built-ins and experiment as you build your workflow. We use a custom Automation here so the pattern is portable—not because custom is always better.

## Learning Objectives

By the end of this mini-lesson you will be able to:
- Create a GitHub PR-triggered Cursor Automation that posts **review comments only** (Bugbot-like)
- Use the **Pull request opened** trigger (ready PRs only—not draft creation)
- Run a Cloud Agent to add a Streamlit Revenue Explorer and walk through draft → ready → automation review
- Launch a **second Cloud Agent** to address review feedback on the **same branch** (separate implementer from reviewer)
- Merge a single PR to `main`, then add **`scripts/check.sh`** (pip dry-run + pytest) and wire the **pre-PR harness** (check script + sub-agent review) into Cloud Agent prompts
- Contrast **custom Automation**, **Bugbot**, product features like **`/babysit`**, and deterministic vs probabilistic gates

## Prerequisites

- Completed [README.md](README.md) setup (fork, clone, local `.venv`, test push)
- Completed [LAB3-Part-1-Cloud-Agent-Environment-Setup.ipynb](LAB3-Part-1-Cloud-Agent-Environment-Setup.ipynb) (env/secrets on your fork)
- Completed [LAB3-Part-2-Running-Cursor-Cloud-Agents.ipynb](LAB3-Part-2-Running-Cursor-Cloud-Agents.ipynb): pipeline fixes merged to `main`, tests green
- Cursor plan with **Automations** enabled; GitHub connected to **your fork**

**Important:** Configure the automation on the **same GitHub repo where you open PRs** (your fork). Each student sets up their own automation, unless the instructor demos on a shared fork.

## Step 1: Understand the workflow

Lab 3 uses a deliberate handoff between humans and agents. You will practice **three review layers**:

| Layer | What | Lab step |
|-------|------|----------|
| **While building** | `pytest` (and later `check.sh`) | Part 2 (`pytest`); Part 3 Step 8+ (`check.sh`) |
| **Before opening a PR** | `check.sh` + **sub-agent** diff review (fresh context) | Step 9 golden prompt; Step 10 demo |
| **After PR is open** | GitHub Automation comments (event-driven reviewer) | Steps 6–6b |

**Lab flow (this lesson):** The steps below are the storyline for Part 3—not a production checklist. We merge the Streamlit PR before adding `check.sh` on purpose so you feel the pytest vs `pip install` gap first (see Step 8).

1. **Cloud Agent** implements code and opens a **draft PR** (Step 4).
2. **You** optionally do detailed local review (Step 5)—this lab offers it **before** agent review; many teams defer until **after**.
3. **You** mark the PR ready and trigger **comments-only** automation review (Step 6).
4. **Optional:** A **second Cloud Agent** addresses blocking review feedback on the **same branch** (Step 6b)—implementer ≠ reviewer.
5. **You** do detailed human review, then merge the Streamlit PR to `main` (Step 7).
6. **You** add `scripts/check.sh` and wire the pre-PR harness into prompts (Steps 8–10), then review and merge the fix PR (Step 11).

### Where teams are headed (north star)

The goal is easy to state: **harness + agents** do as much autonomous work as possible—then mark the PR **ready for review**—and **humans are the final gate** before merge.

That autonomous stretch should include everything cheap and reliable first (**deterministic gates** like `check.sh`, linters, CI), then **tools the agent can call** (including team-specific **MCP servers**—e.g. Slack, Jira—discussed in the course but not set up in this lab), then **fresh-context review** (sub-agent reflection on the diff), and finally **PR-level review** (your custom Automation, **Bugbot**, or similar). The output you want is roughly **one PR** that has already passed those layers and is waiting for a human—not a pile of drafts and fix branches for you to sort out.

Part 3 practices pieces of that stack. You will still click **Ready for review**, run local review, and merge in places where a mature team might automate further. That is intentional: the lab teaches the patterns; wiring them end-to-end takes time. See [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md) for the full framework.

## Step 2: Create the PR review automation

When a Cloud Agent opens a pull request, someone still needs to read the diff, sanity-check dependencies, and leave feedback before you merge. A **Cursor Automation** connects that review step to GitHub: when you mark a draft PR **Ready for review**, an agent runs automatically, reads the PR, and **posts review comments** on GitHub—summary, strengths, blocking issues, and suggestions. You still decide whether to merge; the automation is the **event-driven reviewer** from Step 1 (the outer review layer after the PR exists). In this step you will create that automation on your fork.

**Why build your own?** This step teaches the portable workflow—when to run, what to check, what to post—not the only way to get AI review on PRs. For day-to-day work on Cursor, **Bugbot** is an excellent default; **Approval Agents** are another productized option. Other tools (Claude Code, Copilot, etc.) may offer similar PR review features. Review what your AI provider offers and experiment as you build your workflow. See [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md) Recipe 5.

### Create the automation

1. Open Cursor **Settings** → **Automations** (or the Automations section on [cursor.com](https://cursor.com)). See the [Automations documentation](https://cursor.com/docs/cloud-agent/automations) if the UI differs slightly.
2. Click **Create automation** (or equivalent).
3. **Trigger:** GitHub → **Pull request opened**.
4. **Repository:** Select **your fork** of this starter repo (e.g. `your-username/tdwi-agentic-sales-pipeline-starter`).
5. **Tools / output:** Enable **Comment on pull request** only. You do **not** need tools to open new PRs for this lab.
6. Paste the following into the automation **instructions** field:

```text
Review this pull request.

Focus on:
- Summary of what changed and whether the approach fits the existing codebase
- Correctness, edge cases, and error handling in the diff
- Whether tests were added or updated; note if the PR does not mention test results
- New or changed dependencies (e.g. requirements.txt): necessity and version pinning
- Scope: flag unrelated refactors or drive-by changes
- Security or data-handling concerns if relevant

Post a concise review as PR comments: summary, strengths, blocking errors, suggested improvements. Do not merge or approve.
```

7. Save the automation.

**Why these settings?** In item 3 above, you chose a **trigger**—the GitHub event that starts the automation. Cursor offers two PR-related triggers ([Automations docs](https://cursor.com/docs/cloud-agent/automations)):

| Trigger | When it fires |
|---------|----------------|
| **Draft opened** | A draft PR is created |
| **Pull request opened** | A non-draft PR is created **or** a draft is **marked ready for review** |

**Use Pull request opened** for this lab. Cloud Agents open **draft** PRs while they work; you mark **Ready for review** in Step 6 when you want feedback. If you triggered on **Draft opened** instead, the automation would run as soon as the agent opened the PR—often before you are ready, and again on every new draft.

You also enabled **Comment on pull request** only—no tools to open fix PRs. That is intentional: asking the automation to implement fixes and open new PRs is fragile (wrong base branch, stacked merges, re-fire when you mark a fix PR ready). This lab posts **review comments only**; if you want code changes, you launch a separate implementer agent in **Step 6b**—closer to **Bugbot** plus a human or agent implementer in production. See the debrief for more on auto-fix PRs.

## Step 3: Save and verify the automation

- Confirm the automation appears in your Automations list and is enabled for your fork.
- Use the dashboard **run history** (if available) after Step 6 to confirm it executed.

**Note:** If you already marked a Part 2 PR as ready, toggling ready again may not re-fire the trigger. Part 3’s new PR (Step 4) is the intended test.

## Step 4: Cloud Agent — add Revenue Explorer

Your fork should already have the [`AGENTS.md`](AGENTS.md) you updated in Part 2 on GitHub. The Cloud Agent reads that file automatically—you do **not** need to repeat everything in this prompt.

**Already in `AGENTS.md` (confirm it is pushed to your fork):**
- Repo context and layout (`generate_sales_report.py`, tests, data paths)
- **Testing workflow** — run `python -m pytest test_sales_report.py`, iterate until green before pushing
- **Lab boundaries** — do not modify README or lab notebooks

This prompt states only **what to build** for Part 3. If you changed `AGENTS.md` locally since Part 2, commit and push before starting the agent (same as Part 2 Step 4).

Start a Cloud Agent on **your fork** (same Dockerfile-managed environment as Part 1). Go to [cursor.com/agents](https://cursor.com/agents) or the Agents window in Cursor, select your repository, and paste this prompt:

```text
Add a small Streamlit app called revenue_explorer.py at the repo root.

Product requirements:
- Sidebar: date range filter, multi-select product, optional customer_id filter.
- Main area: KPIs (total revenue, order count, average order value) for the filtered data.
- One chart: daily revenue trend for the filtered data.

Do not duplicate logic. Write clean, well-organized code.
Open a draft PR when done.
```

### Watch the agent and open the PR on GitHub

1. In the [Agents dashboard](https://cursor.com/agents) (or the Agents panel in Cursor), open your session and follow progress until the run **finishes**—edits, terminal output, pytest runs. This may take several minutes.
2. When the agent completes, it will usually open a **draft PR** on your fork. That is expected; you do not need a non-draft PR at this step.
3. On GitHub, open **your fork** → **Pull requests** → the agent's PR (often a `cursor/...` branch into `main`). Confirm it shows **Draft**.
4. Continue to **Step 5** for local review before automation (a lab choice—see Step 5). You will mark the PR **Ready for review** in Step 6 to trigger your automation.

## Step 5: Local review before triggering automation (optional)

**Lab vs practice:** Many teams **defer detailed human review** until after agent review—not one fixed GitHub workflow, but a common pattern: let automated review (CI and agent) run first, then invest human time on the diff. **This lab runs local review before Step 6** to reinforce Part 2 and to surface a common agent mistake early (`pytest` green, `pip install` broken). After the workshop, you might do your detailed checkout and test pass after agent review instead.

1. From the draft PR you opened in Step 4, note the **branch name**.
2. Locally, fetch and check out the agent branch:

```bash
git fetch origin
git checkout <agent-branch-name>
```

3. With `.venv` activated, run tests:

```bash
python -m pytest test_sales_report.py
```

4. **Install dependencies** (this command may **fail**; read the following **warning** first).

   **WARNING**: An agent may pin **Streamlit** to a version that conflicts with pinned **pandas** in `requirements.txt`—**`pytest` can pass while `pip install` fails** with a resolver error. That is the bug we formalize in **Step 8**. If you see it, note it and continue to **Step 6**; you do not need to fix it yet.

```bash
pip install -r requirements.txt
```

5. Run the new app:

```bash
streamlit run revenue_explorer.py
```

6. Read the code changes. Do **not** merge yet—you will trigger the automation in **Step 6**.

**Skipping Step 5?** You may skim the diff on GitHub and continue to **Step 6**, then do detailed local review after agent review (before merge in **Step 7**). You will still hit the dependency lesson in **Step 8** if `pip install` was never run.

## Step 6: Mark the PR ready for review

Return to the **same PR** on GitHub.

1. Scroll to the **bottom** of the PR page and click **Ready for review** (shown on draft PRs only).
2. This fires **Pull request opened**—your review automation should start within a few minutes. Watch run status in the Cursor **Automations** run history (Step 3).

### When the automation finishes

1. Read the automation's **review comments** (summary, strengths, blocking errors, suggestions).
2. If comments flag **blocking** issues you want fixed before merge, continue to **Step 6b**. Otherwise go to **Step 7** for human review and merge.

## Step 6b: Cloud Agent — address review feedback (optional)

**Separate implementer from reviewer:** The automation (reviewer) commented on the PR; a **different** Cloud Agent session should implement fixes—not the same run that built Streamlit.

Skip this step if review comments need no code changes.

1. Note the **PR number** on GitHub (e.g. `#42`).
2. Start a Cloud Agent on **your fork**. Be sure to select the right repo and the new 'streamlit feature' branch that has been created. For the prompt, use the prompt below and fill in the correct PR number.

```text
Read the review comments on PR #<PR-number>. Check out that PR's branch.
Address any blocking issues raised in the review. Push to the same branch—do not open a new PR.
Run python -m pytest test_sales_report.py before pushing.
Do not modify README.md or any lab notebook.
```

3. Watch the agent in the [Agents dashboard](https://cursor.com/agents). When it finishes, the original PR should have new commits.

**WARNING**: The version of **Streamlit** that the agent added may conflict with pinned **pandas** in `requirements.txt`. If so, this step will fail because the environment setup will fail. That is the bug we formalize in **Step 8**.

**Fallback:** If the agent cannot read the PR, paste the automation's blocking findings into the prompt instead.

Continue to **Step 7** to merge.

## Step 7: Human review and merge

Do a **detailed human review** before you merge—whether or not you ran Step 5 earlier. Agent comments (Step 6) are input; you own the merge decision.

1. From your Part 3 Streamlit PR, note the **branch name**.
2. Locally, fetch and check out the branch:

```bash
git fetch origin
git checkout <agent-branch-name>
```


3. With `.venv` activated, run tests:

```bash
python -m pytest test_sales_report.py
```

4. **Install dependencies** (this command may **fail**; read the following **warning** first).

   **WARNING**: The version of **Streamlit** that the agent added may conflict with pinned **pandas** in `requirements.txt`—**`pytest` can pass while `pip install` fails** with a resolver error. That is the bug we formalize in **Step 8**.
   
```bash
pip install -r requirements.txt
```

5. Run `streamlit run revenue_explorer.py`. If Streamlit won't start because `pip install` failed, that's expected—continue to merge and fix the dependency conflict in **Steps 8–11**.

### Merge to `main`

When you are satisfied (including any Step 6b updates), merge on GitHub—same pattern as Part 2.

1. Open your Part 3 Streamlit PR on GitHub.
2. If it is still a **Draft**, click **Ready for review** at the bottom if GitHub requires it before merge.
3. Click **Merge pull request**.
4. Locally, sync your default branch:

```bash
git checkout main
git pull origin main
```

**Dependency check:** If `pip install` failed in Step 5 or Step 7, run `pip install -r requirements.txt` on `main` after the pull—you may see the Streamlit/pandas conflict. **Step 8** adds a script so that check is never optional.

## Step 8: Add `scripts/check.sh` (Recipe 1 starter)

Now, so far you may have run into a bug where the version of streamlit, that the agent added to the environment, is not compatible with the versions of Pandas we have installed. It is possible for an agent to make these kinds of mistakes. It is also possible for an agent to find and fix these mistakes if they are directed to do so. However, it is much cheaper (time and money) if we can catch mistakes with deterministic gates **before** we run more expensive agent review. We also can feed the results of the deterministic check into an agent. This is much more effective than asking an agent to find all the bugs on its own.

This particular kind of error is fairly easy to check. In this lab step, we create an example **check script** that acts as a 'deterministic gate'. In practice, a script like this can run several checks (environment resolution, linting, complexity levels, and test suites). Some of the errors found can be automatically fixed by the same tools that find them, while the other error reports can be used by the agent. Ideally, this script is run before code is pushed; whether it is an engineer working locally or a cloud agent working in the cloud.

**Intentional lab ordering:** In production, this script runs **before** merge (see Step 9). In this lab, we purposely add it **after** Step 7 so you have a chance to experience the gap: `pytest` alone is not enough, and even careful humans may skip `pip install` unless the gate is scripted. Whether you caught the Streamlit/pandas conflict in Step 5, after Step 6, or only on merged `main`, `check.sh` makes the dry-run non-optional.

1. Create the folder and file `scripts/check.sh` at the repo root (same level as `generate_sales_report.py`).
2. Paste the script, show at the bottom of this cell, into the file.
3. Make it executable:

```bash
chmod +x scripts/check.sh
```

3. With `.venv` activated, run from the repo root:

```bash
bash scripts/check.sh
```

If Streamlit and pandas conflict, the **pip dry-run** step should fail with a resolver error—the same failure you may have seen at `pip install` in Step 5. That is the gate working.

```bash
#!/usr/bin/env bash
# TDWI lab — deterministic checks before push/PR (Recipe 1 starter).
# Run from repo root: bash scripts/check.sh

set -euo pipefail
cd "$(dirname "$0")/.."

echo "==> pip dry-run (catch dependency conflicts before install)"
python -m pip install --dry-run -r requirements.txt

# --- Extension points (uncomment as you grow this script) ---
# echo "==> ruff"
# ruff check .
# ruff format --check .

echo "==> pytest"
python -m pytest test_sales_report.py

echo "All checks passed."
```

4. Commit and push `scripts/check.sh` to your fork when it passes (or commit a failing state first if you will fix it in Step 10).

See [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md) Recipe 1 and the fuller templates in [`examples/scripts/`](examples/scripts/) for post-lab ideas.

## Step 9: Wire the pre-PR harness into every Cloud Agent run

In this step, we look at a better example for our original prompt. We now have a low-cost deterministic gate that we instruct the agent to run before committing code. We also instruct the agent to launch a sub-agent to review the code as a final review step. Notice how we are treating the agent like an engineer with tools - not like the entire pipeline itself.


In our lab, step 4 ran **before** `check.sh` existed and without sub-agent review, but that was only for the purpose of the lab. Here is the **golden prompt** for future feature work (Cloud Agent version of local [`/commit-code`](examples/cursor/commands/commit-code.md)—Recipe 3). Note that the "golden" part of this prompt is really that it now follows the `implementer agent - cheap deterministic gates - review agent` pattern, and we are not asking the agent to do everything without any tools. The main principle here is to provide your agent with helpful tools just like you would an engineer. Can a very good engineer find all errors in code just by reading it? Probably not, and, even if they could, it is much more expensive than using linters, test suites, etc...

```text
Add a small Streamlit app called revenue_explorer.py at the repo root.

Product requirements:
- Sidebar: date range filter, multi-select product, optional customer_id filter.
- Main area: KPIs (total revenue, order count, average order value) for the filtered data.
- One chart: daily revenue trend for the filtered data.

Engineering (before opening a PR):
1. Run bash scripts/check.sh and fix all failures. Re-run until exit code 0.
2. Launch a fresh sub-agent to review only the git diff on this branch.
   The reviewer must not be the same context that wrote the changes.
   Focus on scope, correctness, dependencies, and test coverage.
   Fix any blocking issues it reports, then re-run check.sh.
3. Open a draft PR when both gates pass.

Do not duplicate logic. Do not modify README.md or any lab notebook.
```

Sub-agent review is **slow and costly**—same tradeoff as `/commit-code`. Use it as the last automated gate before a PR.

### Optional: `AGENTS.md`

Under **Core Workflow Rules**, add:

```markdown
Before pushing or opening a PR, run `bash scripts/check.sh` and fix all failures. Re-run until exit code 0.
```

Commit and push if you add this (same as Part 2).

### Other surfaces

| Surface | Role |
|---------|------|
| **Agent prompt** | Golden prompt above |
| **`AGENTS.md` / rules** | Repo policy every Cloud Agent reads |
| **Pre-commit / git hooks** | Block commit or push locally |
| **IDE harness hooks** | Cursor, Copilot, Claude Code, etc. |
| **GitHub Actions** (Recipe 4) | Same script on every PR—see [`examples/github/workflows/ci.yml`](examples/github/workflows/ci.yml) |
| **Product features** | **Bugbot** (PR review), **`/babysit`** (PR follow-up)—review and experiment as you build your workflow |

**After the PR is open:** Steps 6–6b (Automation comments + fix agent) are the **outer** review loop—like Bugbot plus a separate implementer. The golden prompt covers the **inner** loop before the PR exists.

## Step 10: Cloud Agent — `check.sh`, sub-agent review, and fix

Let's now launch another cloud agent using this improved workflow. If `check.sh` failed after Step 8 (often a Streamlit/pandas pin conflict), this run should fix that issue. Note that, we are only running this now because we did not use the **Step 9** **golden prompt** originally.

Before you launch this cloud agent, be sure you have committed and pushed the check.sh script we created in **Step 8**.

Paste this prompt:

```text
From the repo root:
1. Run bash scripts/check.sh. Fix failures (especially requirements.txt if pip dry-run reports conflicts). Re-run until green.
2. Launch a fresh sub-agent to review the git diff. The reviewer must not be the same context that wrote the changes. Fix blocking issues it reports, then re-run check.sh.
3. Push to a branch and open a draft PR if changes are not already on a PR. Do not modify README.md or any lab notebook.
```

Watch the agent in the [Agents dashboard](https://cursor.com/agents) as in Step 4. Continue to **Step 11** to review the PR, optionally trigger automation again, merge, and run the app on `main`.

## Step 11: Review, automation pass, merge, and run the app

If the Step 10 agent opened a **new draft PR** (e.g. for `requirements.txt` or `check.sh` fixes), finish the same loop you used earlier—then confirm everything works on `main`. If it only pushed to an **existing** branch and you have not merged yet, treat that PR as in **Step 7**; if you already merged in Step 7, skip to **Merge and run locally** after pulling the latest `main`.

### Review the PR

1. On GitHub, open the Step 10 draft PR.
2. Optionally check out the branch locally, run `bash scripts/check.sh`, and skim the diff—same idea as **Step 7**.

### Trigger automation (optional)

3. If you want another **comments-only** automation pass on this PR, scroll to the bottom and click **Ready for review** (**Step 6**). Read the new comments; use **Step 6b** if you want a fix agent on the same branch.
4. If the PR looks good, skip straight to merge.

### Merge and run locally

5. Merge the Step 10 PR on GitHub when satisfied.
6. Sync `main` locally:

```bash
git checkout main
git pull origin main
```

7. With `.venv` activated, confirm the full stack:

```bash
bash scripts/check.sh
pip install -r requirements.txt
streamlit run revenue_explorer.py
```

You should have a working **Revenue Explorer** on `main`, green checks, and the full Part 3 workflow behind you—from first Streamlit agent through automation review, `check.sh`, and the golden prompt harness.

## Further reading: workflow framework

Parts 1–3 practiced: **probabilistic agents** for implementation, **deterministic scripts** for gates, **sub-agent review** before PRs (inner loop), and **event-driven PR review** after ready (outer loop). Your AI tools may ship helpers for parts of this—**Bugbot**, **`/babysit`**, `/commit-code`—review and experiment as you adopt Recipes 1–6.

The **north star** from Step 1: automate the inner loop until the agent marks a PR ready; you stay the final merge authority. This lab stops short of that full pipeline on purpose—you touched the levers you will combine later.

See [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md) for the full framework, **[Harness vs team workflow](WORKFLOW_RECIPES.md#harness-vs-team-workflow)**, **Recipe 1** (`scripts/check.sh`), Recipes 2–6, and [`examples/`](examples/).


## Debrief questions

1. Why use a **draft PR** for implementation agents and **ready for review** to trigger automation?
2. Why does this lab use **comments-only** automation instead of asking the automation to open fix PRs?
3. What did your automation catch that you would have missed? What did it miss?
4. When would you use a **custom Automation** vs **Bugbot** vs **`/babysit`** vs both?
5. Why use a **second Cloud Agent** (Step 6b) to address review feedback instead of the same session that built the feature?
6. When do you do **detailed** human review—Step 5 (before automation), Step 7 (before merge), or both? What are the tradeoffs?
7. What did `pip install --dry-run` catch that `pytest` alone did not?
8. What is the difference between **sub-agent review before a PR** (Step 9) and **Automation review after ready** (Step 6)?
9. Why run `check.sh` on **every** agent session instead of a separate Cloud Agent only when something breaks?
10. How could you add a **CI completed** trigger so Automations run only after green checks?